In [1]:
import pandas as pd

# Load the dataset
file_path = 'data5001.csv'  # Update with the correct path
data = pd.read_csv(file_path)

# Handle missing values
# Fill missing 'category' with 'Unknown'
data['category'].fillna('Unknown', inplace=True)

# Replace missing 'description' with 'No Description Available'
data['description'].fillna('No Description Available', inplace=True)

# Drop columns with very sparse data
columns_to_drop = ['other_hours', 'reservation_links', 'booking_appointment_link', 'order_links']
data.drop(columns=columns_to_drop, inplace=True)

# Clean and normalize text fields
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = text.replace('\n', ' ').replace('\r', '')  # Remove newlines
    return text.strip()

data['about'] = data['about'].apply(clean_text)
data['description'] = data['description'].apply(clean_text)

# Save the cleaned dataset for further use
data.to_csv('cleaned_data.csv', index=False)

print("Preprocessing complete. Cleaned data saved as 'cleaned_data.csv'.")


Preprocessing complete. Cleaned data saved as 'cleaned_data.csv'.


In [2]:
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# Download VADER lexicon if not already done
nltk.download('vader_lexicon')

# Load the preprocessed dataset
data = pd.read_csv('cleaned_data.csv')

# Initialize VADER sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Function to apply sentiment analysis
def get_sentiment_score(text):
    sentiment = sia.polarity_scores(text)
    return sentiment['compound']  # Compound score as an overall sentiment measure

# Apply sentiment analysis to 'about' and 'description' fields
data['about_sentiment'] = data['about'].apply(get_sentiment_score)
data['description_sentiment'] = data['description'].apply(get_sentiment_score)

# Classify sentiment as positive, negative, or neutral
def classify_sentiment(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

data['about_sentiment_label'] = data['about_sentiment'].apply(classify_sentiment)
data['description_sentiment_label'] = data['description_sentiment'].apply(classify_sentiment)

# Save the sentiment-enhanced dataset
data.to_csv('sentiment_data.csv', index=False)

print("Sentiment analysis complete. Sentiment data saved as 'sentiment_data.csv'.")


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\House\AppData\Roaming\nltk_data...


Sentiment analysis complete. Sentiment data saved as 'sentiment_data.csv'.


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the sentiment-enhanced dataset
data = pd.read_csv('sentiment_data.csv')

# Combine relevant text fields for better feature extraction
data['combined_text'] = data['category'] + ' ' + data['subtypes'] + ' ' + \
                        data['about'] + ' ' + data['description']

# TF-IDF vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['combined_text'])

# Compute cosine similarity between all places
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Function to get recommendations
def get_recommendations(place_name, cosine_sim=cosine_sim):
    # Get the index of the given place
    idx = data[data['name'] == place_name].index[0]

    # Get similarity scores for all places
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort places by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the top 5 similar places
    top_places = sim_scores[1:6]  # Skip the first one (itself)

    # Return place names and similarity scores
    recommendations = [(data.iloc[i[0]]['name'], i[1]) for i in top_places]
    return recommendations

# Example usage
place_name = 'Kiyawat Art Gallery Pune'  # Example place from the dataset
recommendations = get_recommendations(place_name)

print(f"Recommendations for '{place_name}':")
for place, score in recommendations:
    print(f"{place} (Similarity Score: {score:.2f})")


Recommendations for 'Kiyawat Art Gallery Pune':
RAJ ARTS AND REDIUM (Similarity Score: 0.86)
Kakade Arts (Similarity Score: 0.86)
Ss Vastushastra Art Gallery (Similarity Score: 0.86)
Rajlaxmi Photo Frame And Art Gallery (Similarity Score: 0.86)
Beena's Artistry (Similarity Score: 0.86)
